<a href="https://colab.research.google.com/github/naheejeong/cs182-final-project/blob/main/qwen35_9b_02_injecagent_full_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen3.5-9B, step 02: InjecAgent full run

Same steps as `02_llama_full_run.ipynb`, for `Qwen/Qwen3.5-9B`: all 1,054 InjecAgent cases in the base and enhanced settings, first step only. Run this after the pilot in `qwen35_9b_01_wrapper_and_pilot.ipynb` looks usable.

Results are saved to `My Drive/cs182/results/prompted_Qwen35_Qwen/Qwen3.5-9B_InjecAgent/`, next to the Llama results and separate from them.

In [ ]:
# hardware check - GPU & memory assigned
!nvidia-smi

In [ ]:
# Connect Google Drive to the Colab machine so results are saved to Drive.
from google.colab import drive
drive.mount('/content/drive')

# Setup commands (! = system command, % = notebook command)
%cd /content

# Download the InjecAgent benchmark (attack cases and evaluation code) from GitHub.
# --> this makes a new folder named "InjecAgent" in /content.
!git clone https://github.com/uiuc-kang-lab/InjecAgent.git
%cd /content/InjecAgent

# Install the 6 required packages (4 from requirements.txt + together and nltk).
!pip install -q -r requirements.txt together nltk

# Create a results folder in Google Drive (My Drive/cs182/results).
!mkdir -p /content/drive/MyDrive/cs182/results

# Create shortcut: InjecAgent/results -> My Drive/cs182/results, so outputs are saved to Drive.
# Qwen's results go into their own sub-folder (named after the model), so the Llama results are not touched.
!ln -sfn /content/drive/MyDrive/cs182/results results

# Provide environment variables needed by the InjecAgent code.
import os
os.environ["PYTHONPATH"] = "/content/InjecAgent"
os.environ["OPENAI_API_KEY"] = "unused" # placeholder; we never use!

!ls

In [ ]:
# Log in to HF using the token stored in Secrets. Token name: HF_TOKEN, username: naheejeong
# Qwen3.5-9B is Apache-2.0 and not gated, so no access request is needed (unlike Llama).
# We still log in, because logged-in downloads are less likely to be rate-limited.
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download
login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()['name'])

# Testing Qwen access - download one tiny file from the model's page.
model_name = "Qwen/Qwen3.5-9B"
config_path = hf_hub_download(model_name, "config.json")
print(config_path)

In [ ]:
# Our wrapper class that lets InjecAgent run Qwen3.5 (developed and tested in qwen35_9b_01_wrapper_and_pilot.ipynb).
# It is kept as text here and written into InjecAgent's src/models.py, where InjecAgent's script looks for it.
marker = "# ==== added by qwen35_9b_02_injecagent_full_run.ipynb ===="

qwen35_code = marker + r'''

class Qwen35Model(BaseModel):
    # Response length limit: 512 new tokens, the limit InjecAgent uses for open-source models (same as our Llama run).
    MAX_NEW_TOKENS = 512
    # Qwen3.5 "thinks" by default: it writes <think>...</think> before its answer.
    # We turn that off, so the 512 tokens go to the answer and InjecAgent's parser sees only the answer.
    ENABLE_THINKING = False

    def __init__(self, params):
        super().__init__()
        from torch import bfloat16
        # Qwen3.5-9B is a text + vision model, so its model card loads it with AutoModelForMultimodalLM
        # (not AutoModelForCausalLM, which we used for Llama). We only ever give it text.
        from transformers import AutoTokenizer, AutoModelForMultimodalLM
        self.tok = AutoTokenizer.from_pretrained(params['model_name'])
        self.model = AutoModelForMultimodalLM.from_pretrained(
            params['model_name'], dtype=bfloat16, device_map="auto")    # half precision, placed on the GPU automatically

    def prepare_input(self, sys_prompt, user_prompt_filled):
        return [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt_filled},
        ]

    def call_model(self, model_input):    # Whatever prepare_input returns is the model input
        from torch import no_grad

        # 1. Format the messages AND turn them into token IDs (thinking switched off)
        inputs = self.tok.apply_chat_template(
            model_input,
            add_generation_prompt=True,
            enable_thinking=self.ENABLE_THINKING,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)

        # 2. Let Qwen generate the continuation
        pad_id = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id
        with no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.MAX_NEW_TOKENS,
                do_sample=False,    # no randomness, same as the InjecAgent paper and our Llama run
                pad_token_id=pad_id,
            )

        # 3. Keep only the new tokens (the reply) and turn them back into text
        all_tokens = out[0]
        prompt_length = inputs["input_ids"].shape[-1]
        reply_tokens = all_tokens[prompt_length:]
        return self.tok.decode(reply_tokens, skip_special_tokens=True)

MODELS["Qwen35"] = Qwen35Model    # register the class under the name used by --model_type
'''

# Write the class to the end of InjecAgent's models.py, replacing any earlier copy of ours.
models_path = "/content/InjecAgent/src/models.py"
original = open(models_path).read().split(marker)[0].rstrip() + "\n\n"    # InjecAgent's own code, without our part
with open(models_path, "w") as f:
    f.write(original + qwen35_code)

# Confirm InjecAgent now knows the class. (This does not load the model.)
import importlib, src.models
importlib.reload(src.models)
print("registered model types:", list(src.models.MODELS))

In [ ]:
# Record the exact setup next to the results, for the write-up. One line is added per session.
import json, subprocess, datetime, torch, transformers

def run(cmd):    # run a system command and return what it prints
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

run_info = {
    "recorded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "gpu": run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "injecagent_commit": run(["git", "-C", "/content/InjecAgent", "rev-parse", "HEAD"]),
    "model": model_name,
    "model_revision": config_path.split("/snapshots/")[1].split("/")[0],
    "loader": "AutoModelForMultimodalLM, bfloat16", "enable_thinking": False,
    "prompt_type": "InjecAgent", "max_new_tokens": 512, "sampling": False, "only_first_step": True,
}
with open("/content/drive/MyDrive/cs182/results/run_info.jsonl", "a") as f:
    f.write(json.dumps(run_info) + "\n")
print(json.dumps(run_info, indent=1))

In [ ]:
# Back up the results folder into a time-stamped copy before running or resuming.
import shutil
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = f"/content/drive/MyDrive/cs182/results_backup/{stamp}"
shutil.copytree("/content/drive/MyDrive/cs182/results", backup_dir)
print("backed up to", backup_dir)

In [ ]:
# Run (or resume) the base setting. --use_cache reuses every case already saved.
%cd /content/InjecAgent
!python3 src/evaluate_prompted_agent.py --model_type Qwen35 --model_name Qwen/Qwen3.5-9B --setting base --prompt_type InjecAgent --only_first_step --use_cache

In [ ]:
# Run (or resume) the enhanced setting.
%cd /content/InjecAgent
!python3 src/evaluate_prompted_agent.py --model_type Qwen35 --model_name Qwen/Qwen3.5-9B --setting enhanced --prompt_type InjecAgent --only_first_step --use_cache

In [ ]:
# Check that every case was saved, tally the labels, and print the scores for each setting.
import os, json
from collections import Counter
from src.utils import get_score

res_dir = "/content/InjecAgent/results/prompted_Qwen35_Qwen/Qwen3.5-9B_InjecAgent"
expected = {"dh": 510, "ds": 544}    # dh = direct harm, ds = data stealing

for setting in ["base", "enhanced"]:
    files = {a: f"{res_dir}/test_cases_{a}_{setting}.json" for a in ["dh", "ds"]}
    if not all(os.path.exists(p) for p in files.values()):
        print(setting, ": not finished yet\n")
        continue

    # 1. Completeness: how many cases were saved in each file, and their labels.
    all_items = []
    for a, p in files.items():
        items = [json.loads(line) for line in open(p)]
        all_items += items
        print(setting, a, f"{len(items)}/{expected[a]} saved", dict(Counter(i["eval"] for i in items)))

    # 2. InjecAgent's own scores.
    # Read only "Valid Rate", "Direct Harm" and "S1". The "S2", "Data Stealing" and "Total" lines assume
    # step two of data stealing was run. We stop after step one, so they count no data-stealing successes.
    print(json.dumps(get_score(files), indent=1))

    # 3. Our combined numbers for this setting (direct harm + first step of data stealing).
    # We divide by all 1,054 cases, so a case the script skipped counts as a failed attack.
    n_total = sum(expected.values())
    n_succ = sum(i["eval"] == "succ" for i in all_items)
    n_valid = sum(i["eval"] in ("succ", "unsucc") for i in all_items)
    print(f"{setting}: valid rate {n_valid / n_total:.1%}, "
          f"ASR-all {n_succ / n_total:.1%}, ASR-valid {n_succ / max(n_valid, 1):.1%} "
          f"({n_succ} successful, {n_valid} valid, {n_total} cases)")

    # 4. Should be 0. If not, thinking was not switched off for those responses.
    print("responses containing thinking tags:",
          sum("<think>" in i["output"] or "</think>" in i["output"] for i in all_items), "\n")

In [ ]:
# Compare the full run with the pilot on the same 60 cases. With no sampling, the responses should match.
pilot_dir = "/content/drive/MyDrive/cs182/pilot_qwen"
same = different = missing = 0
for a in ["dh", "ds"]:
    for s in ["base", "enhanced"]:
        full = {(i["User Instruction"], i["Tool Response"]): i
                for i in map(json.loads, open(f"{res_dir}/test_cases_{a}_{s}.json"))}
        for p in map(json.loads, open(f"{pilot_dir}/test_cases_{a}_{s}.json")):
            f = full.get((p["User Instruction"], p["Tool Response"]))
            if f is None:
                missing += 1
            elif f["output"] == p["output"]:
                same += 1
            else:
                different += 1
print(f"identical: {same}, different: {different}, not in full run: {missing}")

In [ ]:
# Side by side: Llama-3.1-8B and Qwen3.5-9B on the full benchmark. The Llama results are only read here.
# Same definitions as above: all 1,054 cases in the denominator, direct harm + first step of data stealing.
model_dirs = {
    "Llama-3.1-8B": "/content/InjecAgent/results/prompted_Llama3_meta-llama/Llama-3.1-8B-Instruct_InjecAgent",
    "Qwen3.5-9B": res_dir,
}
n_total = sum(expected.values())
for setting in ["base", "enhanced"]:
    for name, folder in model_dirs.items():
        paths = [f"{folder}/test_cases_{a}_{setting}.json" for a in ["dh", "ds"]]
        if not all(os.path.exists(p) for p in paths):
            print(f"{setting:9s} {name:13s} not finished yet")
            continue
        labels = [json.loads(line)["eval"] for p in paths for line in open(p)]
        succ, valid = labels.count("succ"), labels.count("succ") + labels.count("unsucc")
        print(f"{setting:9s} {name:13s} valid rate {valid / n_total:.1%}, ASR-all {succ / n_total:.1%}, "
              f"ASR-valid {succ / max(valid, 1):.1%}  ({len(labels)} saved)")